In [ ]:
from azure.storage.blob import BlobServiceClient
import pandas as pd 
import numpy as np
import io, os

In [ ]:
# Connect to Azure Blob

os.environ["AZURE_CONNECTION_STRING"] = "ENTER_YOUR_CONNECTION_STRING_HERE"

connection_str = os.getenv("AZURE_CONNECTION_STRING")
client = BlobServiceClient.from_connection_string(connection_str) 
container = client.get_container_client("flight-data") 
 
def load_blob_csv(filename, sep=","):
    blob = container.get_blob_client(filename)
    data = blob.download_blob().readall()
    return pd.read_csv(io.BytesIO(data), sep=sep)

#load data set

df = load_blob_csv("processed/flights_clean.csv")

df.head(1)

In [ ]:
df.shape

In [ ]:
# 1. Time of day buckets (new feature) 
def time_bucket(hour):
    if 5 <= hour < 12:
        return "morning"
    elif 12 <= hour < 17:
        return "afternoon"
    elif 17 <= hour < 21:
        return "evening"
    else:
        return "night"
df["dep_period"] = df["dep_hour"].apply(time_bucket)

In [ ]:
df.head(1)

In [ ]:
# 2. Is weekend?
df["date"] = pd.to_datetime(df["date"], dayfirst=True)

df["day_of_week"] = df["date"].dt.dayofweek

df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

In [ ]:
df.head(1)

In [ ]:
# 3. Route column 
df["route"] = df["from"] + "→" + df["to"]

In [ ]:
df.head(1)

In [ ]:
#  4. convert Stops to numeric values

stop_map = {
    "non-stop": 0,
    "1-stop": 1,
    "2+-stop": 2
}

df["num_stops"] = df["stop"].map(stop_map)

In [ ]:
# 5. One-hot encode categorical columns

cat_cols = ["airline", "from", "to", "dep_period", "class"]

df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

In [ ]:
#6. Select final model features 
drop_cols = ["date", "ch_code", "num_code", "dep_time", "arr_time", "time_taken", "stop", "route"] 
df_model = df_encoded.drop(columns=drop_cols)

In [ ]:
print(f"Final shape: {df_model.shape}")
print(df_model.columns.tolist())

In [ ]:
# Convert dataframe to CSV in memory
csv_buffer = io.StringIO()
df_model.to_csv(csv_buffer, index=False)
csv_content = csv_buffer.getvalue()

# Upload to Azure Blob
blob_client = container.get_blob_client("processed/flights_model_ready.csv")
blob_client.upload_blob(csv_content, overwrite=True)

print("Saved to Azure Blob Storage!")
print("Location: flight-data/processed/flights_model_ready.csv")

In [ ]:
print("Files in Azure Blob Storage:")
for blob in container.list_blobs():
    print(blob.name)